In [7]:
# Load the Kedro IPython extension
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [8]:
# Read parquet file into a DataFrame with polars
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [9]:
# df = pl.read_parquet(
#     "/Users/Carlos_Davalos/Downloads/w_0.8_d_60_o_3.0_t_0.00/second_stage_predictions.parquet"
# ).to_pandas()

df = catalog.load("second_stage_predictions").to_pandas()

df["corrected_predicted_value"] = df.apply(
    lambda row: row["predicted_value"]
    if row["predicted_value"] > row["c_yearly_margin_per_liter"]
    else row["c_yearly_margin_per_liter"],
    axis=1,
)

df["margin_change"] = df["corrected_predicted_value"] - df["c_yearly_margin_per_liter"]


def forced_decile_binning(series: pd.Series, prefix="Q"):
    quantiles = np.linspace(0, 1, 11)  # 10 bins
    bin_edges = series.quantile(quantiles).values
    bin_edges = np.unique(bin_edges)

    if len(bin_edges) <= 2:
        labels = [f"{prefix}1"]
        binned = pd.Series(
            [labels[0]] * len(series), index=series.index, dtype="category"
        )
        mapping = {labels[0]: (series.min(), series.max())}
        return binned, mapping

    num_bins = len(bin_edges) - 1
    labels = [f"{prefix}{i + 1}" for i in range(num_bins)]

    binned = pd.cut(series, bins=bin_edges, labels=labels, include_lowest=True)
    binned = binned.astype("category")
    binned = binned.cat.set_categories(labels, ordered=True)  # ✅ no inplace

    # Build mapping
    mapping = {labels[i]: (bin_edges[i], bin_edges[i + 1]) for i in range(num_bins)}

    return binned, mapping


# Apply to your DataFrame
df["margin_quantile"], quantile_label_map = forced_decile_binning(
    df["corrected_predicted_value"]
)

[06/18/25 19:28:19] INFO     Loading data from second_stage_predictions                         ]8;id=764615;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=968951;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py#403\403]8;;\
                             (PolarsParquetDataset)...                                                             

In [10]:
type(df)

<class 'pandas.core.frame.DataFrame'>

In [11]:
# Stats summary
total = len(df)
top_percent = (df["performance_label"] == "Top Performer").sum() / total * 100
under_percent = (df["performance_label"] == "Under Performer").sum() / total * 100
summary_text = f"✅ Top Performers: {top_percent:.1f}%<br>❌ Under Performers: {under_percent:.1f}%"

# Create vertical subplots (scatter on top, histogram below)
fig = make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.15,
    specs=[[{"type": "scatter"}], [{"type": "xy"}]],
)

# Scatter plot: predicted value
scatter_pred = px.scatter(
    df,
    x="c_yearly_margin_per_liter",
    y="corrected_predicted_value",
    size="c_yearly_network_volumen",
    color="performance_label",
    size_max=20,
)
scatter_pred.update_traces(marker=dict(line=dict(width=0)))
for trace in scatter_pred.data:
    fig.add_trace(trace, row=1, col=1)

# Scatter plot: corrected predicted value
scatter_corr = px.scatter(
    df,
    x="c_yearly_margin_per_liter",
    y="corrected_predicted_value",
    size="c_yearly_network_volumen",
    color="performance_label",
    size_max=20,
)
scatter_corr.update_traces(
    marker=dict(line=dict(width=0), opacity=0.6, symbol="circle-open")
)

# Histogram: margin change
fig.add_trace(
    go.Histogram(
        x=df["margin_change"],
        nbinsx=30,
        marker_color="teal",
        opacity=0.75,
        showlegend=False,
        histnorm="probability",
    ),
    row=2,
    col=1,
)

# Annotations
fig.add_annotation(
    text="Predicted vs Actual Margin",
    x=0.5,
    y=1.08,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=16),
    xanchor="center",
)
fig.add_annotation(
    text="Relative Error Distribution",
    x=0.5,
    y=0.35,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=14),
    xanchor="center",
)

# Performance summary box
fig.add_annotation(
    text=summary_text,
    xref="paper",
    yref="paper",
    x=0.01,
    y=0.96,
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12),
)

# Layout updates
fig.update_layout(
    height=700,
    title_text="Performance Overview",
    margin=dict(t=100, b=60, l=60, r=40),
    showlegend=True,
)

# Axis labels
fig.update_xaxes(title_text="Actual Margin (c_yearly_margin_per_liter)", row=1, col=1)
fig.update_yaxes(title_text="Predicted Margin", row=1, col=1)
fig.update_xaxes(title_text="Relative Margin Change", row=2, col=1)
fig.update_yaxes(title_text="Probability", row=2, col=1)

fig.show()


In [12]:
import dash
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Input, Output, dcc, html
from plotly.subplots import make_subplots

# Dummy data example (replace this with your real `df`)
# df = pd.read_csv("your_data.csv")
# Example structure (you'll replace with actual df)
# df["margin_quantile"] = pd.qcut(df["margin_change"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

# Assuming df is already available
numeric_cols = [col for col in df.select_dtypes(include="number").columns]

app = dash.Dash(__name__)
app.title = "Interactive Dashboard"

# --- 1. Predicted vs. Actual Margin ---
total = len(df)
top_percent = (df["performance_label"] == "Top Performer").sum() / total * 100
under_percent = (df["performance_label"] == "Under Performer").sum() / total * 100
non_regular_percent = (
    (df["performance_label"] == "Non Regular Client").sum() / total * 100
)
impact = sum(df["margin_change"] * df["c_yearly_volumen"]) * 0.0011
impact_millions = impact / 1_000_000

summary_text = (
    f"<b>📊 Total Impact: ${impact_millions:,.2f}M</b><br>"
    f"<b>📊 Total Records: {total}</b><br>"
    f"✅ Top Performers: {top_percent:.1f}%<br>"
    f"❌ Under Performers: {under_percent:.1f}%"
    f"<br>⚠️ Non Regular Clients: {non_regular_percent:.1f}%<br>"
)

scatter_pred = px.scatter(
    df,
    x="c_yearly_margin_per_liter",
    y="corrected_predicted_value",
    size="c_yearly_network_volumen",
    color="performance_label",
    size_max=20,
    hover_data={
        "customer_id": True,
        "c_yearly_margin_per_liter": ":$,.0f",  # Format as currency
        "corrected_predicted_value": ":$,.0f",  # Format as currency, no decimals
        "c_yearly_network_volumen": ":,.0f",  # Comma-separated large numbers
    },
)

scatter_pred.update_layout(
    title="Se Predice el Margen Anual por Litro. En caso de que la prediccion sea menor al margen anual por litro, se corrige a este ultimo.",
    xaxis_title="Margen Anual por Litro (Actual)",
    yaxis_title="Margen Anual por Litro (Prediccion)",
    bargap=0.05,
)

scatter_pred.update_traces(marker=dict(line=dict(width=0)))

scatter_pred.add_annotation(
    text=summary_text,
    xref="paper",
    yref="paper",
    x=0.01,
    y=0.99,
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12),
)

# --- 2. Histogram of Margin Change ---
# Compute min and max with rounding for bin alignment
start = df["margin_change"].min() // 1 * 1
end = df["margin_change"].max() // 1 * 1 + 1

df["margin_binned"] = df["margin_change"].copy()

# Clip all values greater than 20 to exactly 20
df["margin_binned"] = df["margin_binned"].apply(lambda x: 20 if x > 20 else x)

fig_hist = go.Figure()

fig_hist.add_trace(
    go.Histogram(
        x=df["margin_binned"],
        xbins=dict(
            start=df["margin_binned"].min() // 1 * 1,
            end=25,  # Include overflow bin up to 25
            size=1,
        ),
        marker_color="teal",
        opacity=0.75,
        showlegend=False,
    )
)

fig_hist.update_layout(
    title="Cantidad de Clientes por Cambio de Margen (Prediccion - Actual), (Agrupamos >20)",
    xaxis_title="Cambio de Margen (Prediccion - Actual)",
    yaxis_title="Frecuencia",
    bargap=0.05,
)

# --- 3. Volume vs Margin Scatterplot ---
fig_volume_margin = px.scatter(
    df,
    x="c_yearly_network_volumen",
    y="margin_change",
    color="performance_label",
    size_max=20,
    title="Volume vs. Margin Change",
    hover_data={
        "customer_id": True,
        "margin_change": ":$,.0f",  # Format as currency
        "c_yearly_network_volumen": ":,.0f",  # Comma-separated large numbers
    },
)

fig_volume_margin.update_layout(
    title="Evaluacion de Clientes por Volumen Anual vs Cambio de Margen",
    xaxis_title="Volumen Anual (Litros)",
    yaxis_title="Cambio de Margen (Prediccion - Actual)",
    bargap=0.05,
)


# App layout
app.layout = html.Div(
    [
        html.H1("Reporte - Modelo B2B Pricing"),
        html.H2("1. Margen - Prediccion vs Actual"),
        dcc.Graph(figure=scatter_pred),
        html.H2("2. Histograma de Cambio de Margen"),
        dcc.Graph(figure=fig_hist),
        html.H2("3. Volumen (Litros) vs Cambio de Margen"),
        dcc.Graph(figure=fig_volume_margin),
        html.H2(
            "4. Distribucion Variable vs Deciles del Nuevo Margen (con Opcion de Escala Logaritmica)"
        ),
        html.H3("Deciles de Cambio de Margen"),
        html.Table(
            [html.Tr([html.Th("Quantile"), html.Th("Min"), html.Th("Max")])]
            + [
                html.Tr([html.Td(q), html.Td(f"{minv:.2f}"), html.Td(f"{maxv:.2f}")])
                for q, (minv, maxv) in quantile_label_map.items()
            ],
            style={"border": "1px solid black", "margin-bottom": "20px"},
        ),
        html.Div(
            [
                html.Label("Seleccionar Variable:"),
                dcc.Dropdown(
                    id="variable-selector",
                    options=[{"label": col, "value": col} for col in numeric_cols],
                    value=numeric_cols[0],
                ),
                html.Label("Log Scale:"),
                dcc.RadioItems(
                    id="log-toggle",
                    options=[
                        {"label": "Linear", "value": "linear"},
                        {"label": "Log", "value": "log"},
                    ],
                    value="linear",
                    inline=True,
                ),
            ],
            style={"margin": "10px 0"},
        ),
        dcc.Graph(id="boxplot-variable"),
    ]
)


@app.callback(
    Output("boxplot-variable", "figure"),
    Input("variable-selector", "value"),
    Input("log-toggle", "value"),
)
def update_boxplot(variable, scale):
    if scale == "log":
        data = np.log(df[variable].where(df[variable] > 0))
        y_title = f"log({variable})"
    else:
        data = df[variable]
        y_title = variable

    fig = go.Figure()
    fig.add_trace(
        go.Box(
            x=df["margin_quantile"],
            y=data,
            boxmean=True,
            marker_color="steelblue",
        )
    )
    fig.update_layout(
        title=f"Distribucion de {y_title} vs Deciles de Nuevo Margen",
        xaxis_title="Deciles de Nuevo Margen",
        yaxis_title="Valor (log)" if scale == "log" else "Value",
        xaxis=dict(
            categoryorder="array",
            categoryarray=sorted(
                df["margin_quantile"].cat.categories, key=lambda x: int(x[1:])
            ),  # e.g., ["Q1", "Q2", ...]
        ),
    )
    return fig


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=9191)

<IPython.lib.display.IFrame object at 0x30b7b5d10>